In [5]:
import pandas as pd
import numpy as np
df=pd.read_csv("./data/adult.csv")

In [6]:
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [7]:
df=df.replace("?",np.nan)

In [9]:
df.shape

(32561, 15)

In [15]:
from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy="most_frequent")
df["workclass"]=imputer.fit_transform(df[["workclass"]]).ravel()

In [17]:
from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy="most_frequent")
df["occupation"]=imputer.fit_transform(df[["occupation"]]).ravel()

In [18]:
from sklearn.impute import SimpleImputer
imputer=SimpleImputer(strategy="most_frequent")
df["native.country"]=imputer.fit_transform(df[["native.country"]]).ravel()

In [19]:
df.isnull().sum()

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [20]:
X = df.drop(columns=["fnlwgt", "education", "race","income"])


In [21]:
y=df["income"]

In [26]:
X = pd.get_dummies(X)

In [27]:
X.head()

,age,education.num,capital.gain,capital.loss,hours.per.week,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,workclass_Private,workclass_Self-emp-inc,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
0,90,9,0,4356,40,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
1,82,9,0,4356,18,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
2,66,10,0,4356,40,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
3,54,4,0,3900,40,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False
4,41,10,0,3900,40,False,False,False,True,False,...,False,False,False,False,False,False,False,True,False,False


In [42]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [43]:
model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)


In [44]:
model.fit(X_train, y_train)

# Probability predictions
y_prob = model.predict_proba(X_test)[:, 1]

# Lower threshold to improve recall
threshold = 0.35
y_pred = np.where(y_prob >= threshold, ">50K", "<=50K")

In [45]:
from sklearn.metrics import accuracy_score, recall_score, precision_score

print("accuracy:", accuracy_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred, pos_label='>50K'))
print("precision:", precision_score(y_test, y_pred, pos_label='>50K'))

accuracy: 0.7647781360356211
recall: 0.9100765306122449
precision: 0.50638750887154


In [41]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, classification_report, confusion_matrix

# Load data
df = pd.read_csv("./data/adult.csv")

# Replace ? with NaN
df = df.replace("?", np.nan)

# Fill missing values
imputer = SimpleImputer(strategy="most_frequent")
for col in ["workclass", "occupation", "native.country"]:
    df[col] = imputer.fit_transform(df[[col]]).ravel()

# Features and target
X = df.drop(columns=["fnlwgt", "education", "race", "income"])
y = df["income"]

# One-hot encoding
X = pd.get_dummies(X, drop_first=True)

# Train-test split with stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Improved model for better recall
model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

# Probability predictions
y_prob = model.predict_proba(X_test)[:, 1]

# Lower threshold to improve recall
threshold = 0.35
y_pred = np.where(y_prob >= threshold, ">50K", "<=50K")

# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, pos_label=">50K"))
print("Recall   :", recall_score(y_test, y_pred, pos_label=">50K"))
print("F1 Score :", f1_score(y_test, y_pred, pos_label=">50K"))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy : 0.7647781360356211
Precision: 0.50638750887154
Recall   : 0.9100765306122449
F1 Score : 0.6507067943456453

Confusion Matrix:
[[3554 1391]
 [ 141 1427]]

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.96      0.72      0.82      4945
        >50K       0.51      0.91      0.65      1568

    accuracy                           0.76      6513
   macro avg       0.73      0.81      0.74      6513
weighted avg       0.85      0.76      0.78      6513

